# UE24CS352A — Machine Learning Mini-Project
## Machine Learning-Based Prediction of Metabolic Cost During Human Walking

This Google Colab notebook runs the project from start to finish.

### Pipeline
1. Upload and inspect the prepared CSV
2. Remove rows with a missing target
3. Remove a left/right feature only if the two columns are **actually identical**
4. Compare Context, Kinematics + GRF, EMG, and All feature groups
5. Compare regression models using Leave-One-Subject-Out cross-validation
6. Report MSE, RMSE, MAE, and R²
7. Plot results and export tables

**Target:** `metabolic_cost_J_kg_m`

**Important:** This notebook uses the prepared CSV. It does not directly train from the MATLAB `.mat` files. Results are specific to this dataset and evaluation procedure.


## 1. Install / import libraries

In [ ]:
# Colab normally has these libraries installed, but this makes the notebook reproducible.
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")


## 2. Upload the prepared CSV

In [ ]:
print("Upload: metabolic_cost_ml_dataset.csv")
uploaded = files.upload()

csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Please upload the prepared CSV file.")

CSV_PATH = csv_files[0]
print("Using:", CSV_PATH)


## 3. Load and inspect the dataset

In [ ]:
TARGET = "metabolic_cost_J_kg_m"

df_original = pd.read_csv(CSV_PATH)

print("Original shape:", df_original.shape)
print("Columns:")
print(df_original.columns.tolist())

display(df_original.head())


In [ ]:
print("Missing values:")
display(df_original.isna().sum().sort_values(ascending=False).head(15))

print("\nNumber of subjects:", df_original["subject"].nunique())
print("Trials per subject:")
display(df_original.groupby("subject")["trial"].count())


## 4. Clean the target

The supplied dataset contains one missing metabolic-cost value.

Because this is supervised regression, a row without the target cannot be used for training/evaluation, so that row is removed rather than assigning an artificial target value.


In [ ]:
df = df_original.dropna(subset=[TARGET]).copy()

print("Rows before cleaning:", len(df_original))
print("Rows after removing missing target:", len(df))
print("Removed rows:", len(df_original) - len(df))


## 5. Remove only verified exact duplicate columns

Left- and right-side measurements are not automatically duplicates. We remove a right-side column only when its corresponding left-side column exists and the values are identical in every row. Otherwise, both are retained.


In [ ]:
# Only drop a right-side column if it exactly duplicates its matching left-side column.
right_duplicates = []
for right_col in df.columns:
    if "_R_" not in right_col:
        continue
    left_col = right_col.replace("_R_", "_L_")
    if left_col in df.columns and df[left_col].equals(df[right_col]):
        right_duplicates.append(right_col)

print("Verified exact right-side duplicates:", len(right_duplicates))
print(right_duplicates if right_duplicates else "None found; retaining left and right measurements.")

df = df.drop(columns=right_duplicates)
print("Shape after verified duplicate check:", df.shape)


## 6. Define the feature groups

We compare different types of information:

- **Context:** walking speed, incline and trial duration
- **Kinematics + GRF:** context + joint angles, velocities, moments from both sides + ground-reaction-force features
- **EMG:** context + muscle activation features from both sides
- **All:** all feature groups

Both left- and right-side features are included unless a pair was verified to be exactly identical in the previous step.


In [ ]:
context_features = [
    "walking_speed_m_s",
    "incline_code",
    "trial_duration_s"
]

# Include both left and right sides for kinematic measurements.
kin_features = [
    c for c in df.columns
    if c.startswith(("angle_", "velocity_", "moment_"))
]

grf_features = [
    c for c in df.columns
    if c.startswith("grf_")
]

# Include EMG activation features from both sides.
emg_features = [
    c for c in df.columns
    if c.startswith("activation_")
]

# Keep only columns that exist; fail clearly if the expected context columns are missing.
missing_context = [c for c in context_features if c not in df.columns]
if missing_context:
    raise KeyError(f"Expected context columns are missing: {missing_context}")

feature_sets = {
    "Context": context_features,
    "Kinematics+GRF": context_features + kin_features + grf_features,
    "EMG": context_features + emg_features,
    "All": context_features + kin_features + grf_features + emg_features
}

# Guard against accidentally repeated column names in a feature group.
feature_sets = {name: list(dict.fromkeys(cols)) for name, cols in feature_sets.items()}

for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)} features")
    print(cols)
    print()


## 7. Prepare X, y and subject groups

In [ ]:
y = df[TARGET].values
groups = df["subject"].values

print("Samples:", len(df))
print("Subjects:", len(np.unique(groups)))
print("Target:", TARGET)


## 8. Define the regression models

We use:

1. Mean baseline
2. Ridge Regression
3. Support Vector Regression (SVR)
4. Random Forest Regression
5. Gradient Boosting Regression

The mean baseline tells us whether the ML models actually improve over simply predicting the average metabolic cost.


In [ ]:
from sklearn.impute import SimpleImputer

models = {
    "Dummy Mean": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyRegressor(strategy="mean"))
    ]),

    "Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "SVR": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", SVR(C=10, gamma="scale", epsilon=0.1, kernel="rbf"))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=2,
            loss="huber",
            random_state=42
        ))
    ])
}

print("Models ready:", list(models.keys()))


## 9. Leave-One-Subject-Out Cross-Validation

We hold out all trials from one subject at a time. This avoids placing trials from the same subject in both training and test sets.

All imputing and scaling are performed inside each model pipeline, so these steps are learned from the training fold only.


In [ ]:
logo = LeaveOneGroupOut()

results = []
all_predictions = []

for feature_set_name, columns in feature_sets.items():
    X = df[columns].copy()

    for model_name, model in models.items():
        predictions = np.full(len(df), np.nan, dtype=float)

        for train_idx, test_idx in logo.split(X, y, groups):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train = y[train_idx]

            model.fit(X_train, y_train)
            predictions[test_idx] = model.predict(X_test)

        mse = mean_squared_error(y, predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y, predictions)
        r2 = r2_score(y, predictions)

        results.append({
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2
        })

        for i in range(len(df)):
            all_predictions.append({
                "Feature_Set": feature_set_name,
                "Model": model_name,
                "subject": df.iloc[i]["subject"],
                "trial": df.iloc[i]["trial"],
                "actual": y[i],
                "predicted": predictions[i],
                "error": y[i] - predictions[i]
            })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
predictions_df = pd.DataFrame(all_predictions)

display(results_df.round(4))


## 10. Find the best model

In [ ]:
non_dummy = results_df[results_df["Model"] != "Dummy Mean"].copy()

best = non_dummy.iloc[0]

print("BEST CONFIGURATION")
print("------------------")
print("Feature set :", best["Feature_Set"])
print("Model       :", best["Model"])
print("RMSE        :", round(best["RMSE"], 4))
print("MAE         :", round(best["MAE"], 4))
print("R²          :", round(best["R2"], 4))


## 11. Compare the best feature set for every model

In [ ]:
best_per_model = (
    non_dummy.sort_values("RMSE")
    .groupby("Model", as_index=False)
    .first()
    .sort_values("RMSE")
)

display(best_per_model[[
    "Model", "Feature_Set", "RMSE", "MAE", "R2"
]].round(4))


## 12. Model comparison plot

In [ ]:
plt.figure(figsize=(9, 5))

plot_df = best_per_model.sort_values("RMSE")

plt.bar(plot_df["Model"], plot_df["RMSE"])
plt.ylabel("Leave-One-Subject-Out RMSE")
plt.xlabel("Model")
plt.title("Best Feature Set Performance by Model")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 13. Target distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df[TARGET], bins=12)

plt.xlabel("Metabolic Cost (J/kg/m)")
plt.ylabel("Number of Samples")
plt.title("Distribution of Metabolic Cost")
plt.tight_layout()
plt.show()


## 14. Correlation heatmap

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, xticklabels=True, yticklabels=True)

plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()


## 15. Actual vs predicted

In [ ]:
best_predictions = predictions_df[
    (predictions_df["Feature_Set"] == best["Feature_Set"]) &
    (predictions_df["Model"] == best["Model"])
].copy()

plt.figure(figsize=(7, 6))

plt.scatter(
    best_predictions["actual"],
    best_predictions["predicted"]
)

low = min(
    best_predictions["actual"].min(),
    best_predictions["predicted"].min()
)

high = max(
    best_predictions["actual"].max(),
    best_predictions["predicted"].max()
)

plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Actual Metabolic Cost")
plt.ylabel("Predicted Metabolic Cost")
plt.title(f"Actual vs Predicted — {best['Model']} + {best['Feature_Set']}")
plt.tight_layout()
plt.show()


## 16. Feature importance

For interpretation only, fit a Gradient Boosting model on all cleaned rows and inspect its feature importance. Because this fit uses the complete dataset, its importance plot is descriptive and is not a separate test-set result. The reported performance numbers above come from subject-wise cross-validation.


In [ ]:
all_features = feature_sets["All"]

# Fit the imputer and Gradient Boosting model for exploratory feature importance.
importance_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=2,
        loss="huber",
        random_state=42
    ))
])

importance_pipeline.fit(df[all_features], y)
importance_model = importance_pipeline.named_steps["model"]
importance = pd.Series(
    importance_model.feature_importances_,
    index=all_features
).sort_values(ascending=False)

display(importance.head(15).to_frame("importance"))

plt.figure(figsize=(9, 6))
importance.head(12).sort_values().plot(kind="barh")
plt.xlabel("Feature Importance")
plt.title("Top Features — Gradient Boosting (Exploratory)")
plt.tight_layout()
plt.show()


## 17. Save all results

In [ ]:
OUTPUT_DIR = Path("/content/metabolic_cost_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_DIR / "cleaned_dataset.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "cross_validated_predictions.csv", index=False)
importance.to_csv(OUTPUT_DIR / "feature_importance.csv", header=["importance"])

print("Results saved to:", OUTPUT_DIR)
print([p.name for p in OUTPUT_DIR.iterdir()])


## 18. Download the results

In [ ]:
# Create a ZIP containing the generated result files.
import shutil

zip_file = shutil.make_archive(
    "/content/metabolic_cost_results",
    "zip",
    OUTPUT_DIR
)

print("Created:", zip_file)

files.download(zip_file)


# Final interpretation

The notebook selects the best non-baseline configuration automatically using the lowest Leave-One-Subject-Out RMSE. **Use the values printed by your latest run in your report and presentation.** Do not reuse result numbers from an earlier notebook run after changing the feature groups or preprocessing.

### Viva explanation

> We treat metabolic-cost prediction as a supervised regression problem. We remove rows with a missing target, retain left/right measurements unless a pair is proven identical, compare different feature groups and regression models, and evaluate them with Leave-One-Subject-Out cross-validation. We report MSE, RMSE, MAE and R² because the target is continuous.
